# Tutorial 5: Random Generation → High-T Equilibration → Batch Quench

This tutorial demonstrates a **hybrid workflow** that combines random structure
generation with melt-quench sampling to produce multiple independent amorphous
TiO₂ structures:

1. **Generate** a random cubic TiO₂ structure (24 atoms, 3.2 g/cm³)
2. **Optimise** with CHGNet (fixed volume — preserves target density)
3. **Equilibrate** at 2000 K for 20 ps (NVT) — creates a well-sampled liquid
4. **Extract** 10 decorrelated snapshots from the high-T trajectory
5. **Batch quench** each snapshot independently (NVT): quench → low-T eq → final opt (Stages 5 → 6 → 7)

This approach is efficient because:
- Random placement + optimisation gives a reasonable starting liquid (skips slow heating)
- NVT ensemble preserves the target amorphous density throughout
- One long high-T equilibration is shared across all structures
- Each snapshot is quenched independently → statistically independent amorphous structures

**Estimated runtime:** ~15–20 min on CPU with CHGNet

In [ ]:
# Run this notebook from its own tutorial directory.
# Shared diagnostics live one directory above, in Tutorials/.
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))
from analysis_helpers import (get_density, minimum_image_rdf, coordination_numbers)

import os
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write
from ase.data import atomic_masses, atomic_numbers

---
## 3.1 Generate a random TiO₂ structure

We generate a single cubic TiO₂ cell with 24 atoms (8 Ti + 16 O)
at a target density of 3.5 g/cm³.

In [ ]:
from amorphgen.pipeline.random_gen import generate_random

composition = {"Ti": 8, "O": 16}  # 24 atoms, TiO2 stoichiometry

atoms = generate_random(
    composition=composition,
    target_density=3.2,             # g/cm³ (amorphous TiO2 density)
    minsep={
        "Ti-Ti": 2.5,
        "Ti-O":  1.6,
        "O-O":   2.2,
    },
    seed=42,
    max_attempts_per_atom=50000,
)

print(f"Formula:  {atoms.get_chemical_formula()}")
print(f"Atoms:    {len(atoms)}")
print(f"Cell:     {atoms.cell[0,0]:.3f} x {atoms.cell[1,1]:.3f} x {atoms.cell[2,2]:.3f} Å")

density = get_density(atoms)
print(f"Density:  {density:.3f} g/cm³ (target: 3.2)")

write("TiO2_random.extxyz", atoms, format="extxyz")
print("\nSaved -> TiO2_random.extxyz")


---
## 3.2 Optimise the random structure

Relax the random structure to remove unphysical overlaps and
bring it to a local energy minimum. We optimise **atomic positions only**
(no cell relaxation) to preserve the target density of 3.2 g/cm³.

In [ ]:
from amorphgen.utils import get_calculator
from ase.optimize import LBFGS
from ase.io import write

calc = get_calculator(model="chgnet", device="cpu")
print("CHGNet calculator loaded.")

# Optimise positions only (no cell filter) to preserve target density
atoms.calc = calc
opt = LBFGS(atoms, logfile="TiO2_opt.log")
opt.run(fmax=0.1, steps=200)

e_per_atom = atoms.get_potential_energy() / len(atoms)
write("TiO2_optimised.extxyz", atoms, format="extxyz")
write("TiO2_optimised.cif", atoms)
print(f"\nOptimised energy: {e_per_atom:.4f} eV/atom")
print(f"Saved -> TiO2_optimised.extxyz, TiO2_optimised.cif")

optimised = atoms.copy()

---
## 3.3 Equilibrate at 2000 K (NVT, 20 ps)

Run NVT MD at 2000 K to create a well-sampled disordered liquid.
NVT keeps the volume (and hence density) fixed at our target value.
The trajectory will be used to extract decorrelated snapshots.

In [ ]:
from amorphgen.pipeline.equilibrate import run as eq_run

liquid = eq_run(
    optimised,
    cfg_override={
        "eq_high": {
            "ensemble": "NVT",
            "T": 2000,
            "steps": 10000,            # 20 ps at 2 fs timestep
            "timestep": 2.0,
            "friction": 0.01,
            "log_file": "TiO2_eq_2000K.log",
            "traj_file": "TiO2_eq_2000K.extxyz",
            "output_xyz": "TiO2_eq_2000K_final.extxyz",
        },
    },
    calc=calc,
    stage="high",
)

print(f"Equilibrated liquid: {len(liquid)} atoms")

---
## 3.4 Extract 10 snapshots from the trajectory

We select 10 evenly spaced frames from the high-T trajectory.
These are decorrelated liquid configurations that will be quenched
independently.

In [ ]:
# Read the full trajectory
traj = read("TiO2_eq_2000K.extxyz", index=":")
print(f"Total frames in trajectory: {len(traj)}")

# Select 10 evenly spaced snapshots (skip first 10% for thermalisation)
n_snapshots = 5
start_frame = len(traj) // 10          # skip first 10%
indices = np.linspace(start_frame, len(traj) - 1, n_snapshots, dtype=int)

snapshots_dir = "snapshots"
os.makedirs(snapshots_dir, exist_ok=True)

snapshot_files = []
for i, idx in enumerate(indices):
    fname = f"{snapshots_dir}/snapshot_{i:04d}.extxyz"
    write(fname, traj[idx], format="extxyz")
    snapshot_files.append(fname)
    print(f"  Snapshot {i}: frame {idx} -> {fname}")

print(f"\n{n_snapshots} snapshots saved to {snapshots_dir}/")

---
## 3.5 Batch quench: Stages 5 → 6 → 7 on each snapshot

Each snapshot is independently:
- **Stage 5:** Quenched from 2000 → 300 K (NVT, ~2000 K/ps)
- **Stage 6:** Equilibrated at 300 K (NVT, 4 ps)
- **Stage 7:** Final optimisation

NVT preserves the cell volume throughout, maintaining our target density.
This produces 10 statistically independent amorphous TiO₂ structures.

In [ ]:
from amorphgen import MeltQuenchPipeline

amorphous_structures = []

for i, snap_file in enumerate(snapshot_files):
    run_dir = f"batch_quench/run_{i:04d}"
    print(f"\n{'='*60}")
    print(f"  Run {i+1}/{n_snapshots}: {snap_file} -> {run_dir}")
    print(f"{'='*60}")

    pipe = MeltQuenchPipeline(
        input_file=snap_file,
        work_dir=run_dir,
        cfg_override={
            "model": "chgnet",
            "device": "cpu",

            "opt": {
                "fmax": 0.1,
                "max_steps": 100,
            },

            # Stage 5: quench (NVT, 2000 → 300 K, ~2000 K/ps)
            "quench": {
                "ensemble": "NVT",
                "T_start": 2000,
                "T_end": 300,
                "T_step": -200,
                "steps_per_T": 50,
                "timestep": 2.0,
                "friction": 0.01,
            },

            # Stage 6: low-T equilibration (NVT, 300 K, 4 ps)
            "eq_low": {
                "ensemble": "NVT",
                "T": 300,
                "steps": 2000,
                "timestep": 2.0,
                "friction": 0.01,
            },
        },
    )

    final = pipe.run(stages=[5, 6, 7])
    amorphous_structures.append(final.copy())

print(f"\n\nDone! {len(amorphous_structures)} amorphous structures generated.")

---
## 3.6 Analyse the amorphous ensemble

### RDF across all 10 structures

In [ ]:
# Preserve this tutorial’s grid and minimum-image convention.
def compute_rdf(atoms, rmax=6.0, nbins=150):
    return minimum_image_rdf(atoms, rmax=rmax, nbins=nbins)

fig, ax = plt.subplots(figsize=(8, 5))

for i, atoms in enumerate(amorphous_structures):
    r, g = compute_rdf(atoms)
    ax.plot(r, g, color="orangered", alpha=0.2, lw=0.8)

# Mean RDF
rdfs = [compute_rdf(a)[1] for a in amorphous_structures]
r = compute_rdf(amorphous_structures[0])[0]
mean_rdf = np.mean(rdfs, axis=0)
ax.plot(r, mean_rdf, color="orangered", lw=2, label=f"Mean (n={len(amorphous_structures)})")

ax.set_xlabel("r (Å)")
ax.set_ylabel("g(r)")
ax.set_title("Amorphous TiO₂ — RDF ensemble", fontweight="bold")
ax.set_xlim(0, 6)
ax.legend()
plt.tight_layout()
plt.savefig("rdf_ensemble_TiO2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved -> rdf_ensemble_TiO2.png")

### Density distribution

In [ ]:
densities = [get_density(a) for a in amorphous_structures]

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(range(len(densities)), densities, color="orangered", edgecolor="black", s=60)
ax.axhline(np.mean(densities), color="black", ls="--", lw=1.5,
           label=f"Mean: {np.mean(densities):.3f} g/cm³")
ax.axhline(3.84, color="steelblue", ls=":", lw=1, label="Crystal (anatase): 3.84")
ax.set_xlabel("Structure index")
ax.set_ylabel("Density (g/cm³)")
ax.set_title("Density of 10 amorphous TiO₂ structures")
ax.legend()
plt.tight_layout()
plt.savefig("density_ensemble_TiO2.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Mean density: {np.mean(densities):.3f} ± {np.std(densities):.3f} g/cm³")
print(f"Experimental amorphous TiO₂: ~3.4–3.6 g/cm³")

### Coordination number (Ti–O)

In crystalline TiO₂ (rutile/anatase), Ti is 6-coordinated by O.
Amorphous TiO₂ typically shows a mix of 4-, 5-, and 6-fold coordination.

In [ ]:
cutoff_TiO = 2.5  # Å — first coordination shell

cn_all = []
for atoms in amorphous_structures:
    cn = coordination_numbers(atoms, "Ti", "O", cutoff_TiO)
    cn_all.extend(cn)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(cn_all, bins=range(0, 10), align="left",
        color="orangered", edgecolor="white", alpha=0.8)
ax.axvline(6, ls="--", color="steelblue", label="Crystal CN=6")
ax.set_xlabel("CN (Ti–O)")
ax.set_ylabel("Count")
ax.set_title(f"Ti–O coordination number (cutoff {cutoff_TiO} Å)")
ax.legend()
ax.set_xlim(-0.5, 9.5)
plt.tight_layout()
plt.savefig("cn_ensemble_TiO2.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Mean CN: {np.mean(cn_all):.2f} ± {np.std(cn_all):.2f}")
print(f"Crystal: CN = 6 (octahedral)")

### Energy distribution

In [ ]:
# Re-attach calculator to get energies
for a in amorphous_structures:
    a.calc = calc

energies = [a.get_potential_energy() / len(a) for a in amorphous_structures]
mean_e = np.mean(energies)
std_e = np.std(energies)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(range(len(energies)), energies, color="orangered", edgecolor="black", s=60)
ax.axhline(mean_e, color="black", ls="--", lw=1.5, label=f"Mean: {mean_e:.4f} eV/atom")
ax.fill_between([-1, len(energies)], mean_e - std_e, mean_e + std_e,
                color="gray", alpha=0.1, label=f"Std: {std_e:.4f}")
ax.set_ylim(np.min(energies) - 0.02, np.max(energies) + 0.02)
ax.set_xlabel("Structure index")
ax.set_ylabel("Energy (eV/atom)")
ax.set_title("Energy of 10 amorphous TiO₂ structures")
ax.legend()
plt.tight_layout()
plt.savefig("energy_ensemble_TiO2.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Note: CLI equivalent

This workflow can also be done from the command line:

```bash
# Step 1: Generate random structure
amorphgen --random-gen \
    --composition Ti=8,O=16 \
    --target-density 3.2 \
    --n-structures 1 \
    --model chgnet \
    --device cpu \
    --work-dir random_TiO2

# Step 2-4: Run stages 1-4 (optimise + premelt + melt + high-T eq)
amorphgen random_TiO2/TiO2_000.extxyz \
    --model chgnet \
    --device cpu \
    --stages 1 4 \
    --eq-high-ensemble NVT \
    --eq-high-T 2000 \
    --eq-high-steps 10000 \
    --work-dir melt_run

# Step 5: Batch quench from snapshots (NVT)
amorphgen --batch-quench \
    --snapshot-dir snapshots/ \
    --n-runs 10 --select uniform \
    --batch-stages 5 6 7 \
    --quench-ensemble NVT \
    --eq-low-ensemble NVT \
    --model chgnet \
    --device cpu \
    --work-dir batch_quench
```

In [ ]:
print(f"{'Property':<25s}  {'Mean':>10s}  {'Std':>10s}  {'Crystal':>10s}")
print("-" * 60)
print(f"{'Density (g/cm³)':<25s}  {np.mean(densities):10.3f}  {np.std(densities):10.3f}  {'3.84':>10s}")
print(f"{'Energy (eV/atom)':<25s}  {mean_e:10.4f}  {std_e:10.4f}  {'—':>10s}")
print(f"{'Ti–O CN':<25s}  {np.mean(cn_all):10.2f}  {np.std(cn_all):10.2f}  {'6.0':>10s}")
print(f"\n10 independent amorphous TiO₂ structures generated.")
print(f"Structures saved in batch_quench/run_NNNN/ directories.")

---
## Note: CLI equivalent

This workflow can also be done from the command line:

```bash
# Step 1: Generate random structure
amorphgen --random-gen \
    --composition Ti=8,O=16 \
    --target-density 3.5 \
    --n-structures 1 \
    --model chgnet \
    --device cpu \
    --work-dir random_TiO2

# Step 2-4: Run stages 1-4 (optimise + premelt + melt + high-T eq)
amorphgen random_TiO2/TiO2_000.extxyz \
    --model chgnet \
    --device cpu \
    --stages 1 4 \
    --eq-high-ensemble NPT \
    --eq-high-T 2000 \
    --eq-high-steps 10000 \
    --work-dir melt_run

# Step 5: Batch quench from snapshots
amorphgen --batch-quench \
    --snapshot-dir snapshots/ \
    --n-runs 10 --select uniform \
    --batch-stages 5 6 7 \
    --model chgnet \
    --device cpu \
    --work-dir batch_quench
```

---
## Output summary

After running this tutorial, you should have:

```
TiO2_random.extxyz              # random starting structure
TiO2_optimised.cif              # optimised structure
TiO2_eq_2000K.extxyz            # high-T equilibration trajectory
TiO2_eq_2000K_final.extxyz      # final liquid snapshot
snapshots/
├── snapshot_0000.extxyz        # 10 decorrelated liquid snapshots
├── snapshot_0001.extxyz
├── ...
└── snapshot_0009.extxyz
batch_quench/
├── run_0000/                   # 10 independent quench runs
│   ├── stage5_quench.extxyz
│   ├── stage6_eq.extxyz
│   └── opt_final.cif           # amorphous structure
├── run_0001/
├── ...
└── run_0009/
rdf_ensemble_TiO2.png
density_ensemble_TiO2.png
cn_ensemble_TiO2.png
energy_ensemble_TiO2.png
```

Open any `opt_final.cif` in VESTA to visualise the amorphous structures.